# Building a clean, usable dataset with a defined target (Pipeline Construction)

In [51]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

In [52]:
# Base paths
BASE_PATH = Path("../data/raw")

ECOM_PATH = BASE_PATH / "Brazilian e-Commerce"
FUNNEL_PATH = BASE_PATH / "Marketing Funnel"

In [53]:
# Create SQLite connection
conn = sqlite3.connect("../data/processed/olist.db")

In [54]:
# Load core datasets
orders = pd.read_csv(ECOM_PATH / "olist_orders_dataset.csv")
customers = pd.read_csv(ECOM_PATH / "olist_customers_dataset.csv")
reviews = pd.read_csv(ECOM_PATH / "olist_order_reviews_dataset.csv")
items = pd.read_csv(ECOM_PATH / "olist_order_items_dataset.csv")
products = pd.read_csv(ECOM_PATH / "olist_products_dataset.csv")

In [55]:
# Load marketing datasets
leads = pd.read_csv(FUNNEL_PATH / "olist_marketing_qualified_leads_dataset.csv")
deals = pd.read_csv(FUNNEL_PATH / "olist_closed_deals_dataset.csv")

In [56]:
print(orders.shape)
print(customers.shape)
print(reviews.shape)

(99441, 8)
(99441, 5)
(99224, 7)


In [57]:
orders.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [58]:
# Load dataframes into SQLite database
orders.to_sql("orders", conn, if_exists="replace", index=False)
customers.to_sql("customers", conn, if_exists="replace", index=False)
reviews.to_sql("reviews", conn, if_exists="replace", index=False)
items.to_sql("items", conn, if_exists="replace", index=False)
products.to_sql("products", conn, if_exists="replace", index=False)

leads.to_sql("leads", conn, if_exists="replace", index=False)
deals.to_sql("deals", conn, if_exists="replace", index=False)

842

In [59]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
pd.read_sql(query, conn)

,name
0,orders
1,customers
2,reviews
3,items
4,products
5,leads
6,deals


In [60]:
orders.columns
customers.columns
reviews.columns

Index(['review_id', 'order_id', 'review_score', 'review_comment_title',
       'review_comment_message', 'review_creation_date',
       'review_answer_timestamp'],
      dtype='object')

The core relationships

1. Orders → Customers
orders.customer_id = customers.customer_id
2. Customers → Real person
customers.customer_unique_id = real person
3. Reviews → Orders
reviews.order_id = orders.order_id

For:
- Order-level analysis (main goal)
👉 Use order_id
- Customer behavior / repeat purchase (later)
👉 Use customer_unique_id

### The dataset distinguishes between transactional customer IDs and persistent customer identities, allowing analysis at both order and customer levels.

In [61]:
# SQL query to join orders, customers, and reviews
query = """
SELECT
    o.order_id,
    o.customer_id,
    c.customer_unique_id,
    o.order_purchase_timestamp,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    r.review_score,
    r.review_comment_message,
    r.review_comment_title
FROM orders o
LEFT JOIN customers c 
    ON o.customer_id = c.customer_id
LEFT JOIN reviews r 
    ON o.order_id = r.order_id
"""

In [62]:
df = pd.read_sql(query, conn)

In [63]:
df.isnull().sum()

order_id                             0
customer_id                          0
customer_unique_id                   0
order_purchase_timestamp             0
order_delivered_customer_date     2987
order_estimated_delivery_date        0
review_score                       768
review_comment_message           59015
review_comment_title             88424
dtype: int64

1. Missing delivery date (2987) / These are likely:

- orders not delivered yet
- cancelled orders
- failed deliveries

**These are not valid “completed experiences”**

2. Missing review score (768) / These are:

- customers who didn’t leave a review

**Important nuance:**

- Not dissatisfaction
- Not satisfaction
- Just no explicit feedback signal

### Updated approach

For this project, I kept all completed delivered orders, including those without review scores.

This makes the dataset more realistic for customer experience analysis, because in real business settings many customers never leave explicit feedback. Instead of dropping those orders, I separate:

- whether a customer left a review (`has_review`)
- whether a reviewed order was dissatisfied (`is_dissatisfied`)

This allows the analysis to distinguish between **observed dissatisfaction** and **silent customers**, which creates a richer post-purchase experience view.

In [64]:
# Drop rows with missing delivery dates
df = df.dropna(subset=["order_delivered_customer_date"])

In [65]:
df.isnull().sum()
df.shape

(97005, 9)

In [66]:
# Convert date columns to datetime
df["order_purchase_timestamp"] = pd.to_datetime(df["order_purchase_timestamp"])
df["order_delivered_customer_date"] = pd.to_datetime(df["order_delivered_customer_date"])
df["order_estimated_delivery_date"] = pd.to_datetime(df["order_estimated_delivery_date"])

In [67]:
# Delivery delay
df["delivery_delay"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

In [68]:
# Additional expectation-gap feature
df["delivery_deviation_abs"] = df["delivery_delay"].abs()

In [69]:
# Feedback availability + dissatisfaction target
df["has_review"] = df["review_score"].notna().astype(int)

df["is_dissatisfied"] = np.where(
    df["review_score"].notna(),
    np.where(df["review_score"] <= 2, 1, 0),
    np.nan
)

In [70]:
# Review participation check
print("Review rate:", df["has_review"].mean())

# Delivery delay among reviewed orders only
df[df["has_review"] == 1].groupby("is_dissatisfied")["delivery_delay"].mean()

Review rate: 0.9933405494562136


is_dissatisfied
0.0   -12.912736
1.0    -5.149879
Name: delivery_delay, dtype: float64

In [71]:
# Compare reviewed vs non-reviewed orders
df.groupby("has_review")[["delivery_delay", "delivery_deviation_abs"]].mean()

,delivery_delay,delivery_deviation_abs
has_review,,
0,-6.360681,13.400929
1,-11.917797,13.318330


In [72]:
df.groupby("has_review")[["review_score"]].count()

,review_score
has_review,
0,0
1,96359


In [73]:
df.groupby("has_review")[["order_id"]].count()

,order_id
has_review,
0,646
1,96359


In [74]:
df.groupby("has_review")[["delivery_delay", "delivery_deviation_abs"]].mean()

,delivery_delay,delivery_deviation_abs
has_review,,
0,-6.360681,13.400929
1,-11.917797,13.318330


In [75]:
order_values = items.groupby("order_id").agg({
    "price": "sum",
    "freight_value": "sum",
    "order_item_id": "count"
}).reset_index()

order_values["order_value"] = order_values["price"] + order_values["freight_value"]
order_values.rename(columns={"order_item_id": "product_count"}, inplace=True)

order_values = order_values[["order_id", "order_value", "product_count"]]

In [76]:
df = df.merge(order_values, on="order_id", how="left")

In [77]:
# Order complexity / pricing features
df["avg_price_per_item"] = df["order_value"] / df["product_count"]

# Time-based features
df["purchase_month"] = df["order_purchase_timestamp"].dt.month
df["purchase_dow"] = df["order_purchase_timestamp"].dt.dayofweek

In [78]:
# Compare reviewed vs non-reviewed customers across key order features (allows us to see whether 'silent'customers differ meaningfully)
df.groupby("has_review")[["order_value", "product_count", "avg_price_per_item"]].mean()

,order_value,product_count,avg_price_per_item
has_review,,,
0,198.793251,1.280186,174.484089
1,159.410276,1.141689,144.999596


In [79]:
# Create delay buckets
def delay_bucket(x):
    if x <= -3:
        return "Very Early"
    elif -3 < x <= 0:
        return "Slightly Early"
    elif 0 < x <= 3:
        return "On Time"
    else:
        return "Late"

df["delay_bucket"] = df["delivery_delay"].apply(delay_bucket)

In [80]:
# Sort by customer and purchase date to create historical customer features
df = df.sort_values(["customer_unique_id", "order_purchase_timestamp"])

df["prior_orders"] = df.groupby("customer_unique_id").cumcount()
df["is_repeat_customer"] = (df["prior_orders"] > 0).astype(int)

In [81]:
df[[
    "order_value",
    "product_count",
    "avg_price_per_item",
    "delivery_delay",
    "delivery_deviation_abs",
    "prior_orders"
]].describe()

,order_value,product_count,avg_price_per_item,delivery_delay,delivery_deviation_abs,prior_orders
count,97005.000000,97005.000000,97005.000000,97005.000000,97005.00000,97005.000000
mean,159.672545,1.142611,145.195947,-11.880790,13.31888,0.049564
std,218.444711,0.540026,196.537091,10.183992,8.21425,0.297157
min,9.590000,1.000000,9.341429,-147.000000,0.00000,0.000000
25%,61.810000,1.000000,57.430000,-17.000000,8.00000,0.000000
50%,105.220000,1.000000,95.800000,-12.000000,13.00000,0.000000
75%,176.200000,1.000000,162.480000,-7.000000,17.00000,0.000000
max,13664.080000,21.000000,6929.310000,188.000000,188.00000,14.000000


In [82]:
df.head()

,order_id,customer_id,customer_unique_id,order_purchase_timestamp,order_delivered_customer_date,order_estimated_delivery_date,review_score,review_comment_message,review_comment_title,delivery_delay,...,has_review,is_dissatisfied,order_value,product_count,avg_price_per_item,purchase_month,purchase_dow,delay_bucket,prior_orders,is_repeat_customer
51501,e22acc9c116caa3f2b7121bbb380d08e,fadbb3709178fc513abc1b2670aa1ad2,0000366f3b9a7992bf8c76cfdf3221e2,2018-05-10 10:56:27,2018-05-16 20:48:37,2018-05-21,5.0,"Adorei a cortina, ficou linda na minha sala, e...",Super Recomendo,-5,...,1,0.0,141.90,1,141.90,5,3,Very Early,0,0
72082,3594e05a005ac4d06a72673270ef9ec9,4cb282e167ae9234755102258dd52ee8,0000b849f77a49e4a4ce2b2a4ca5be3f,2018-05-07 11:11:27,2018-05-10 18:02:42,2018-05-15,4.0,None,None,-5,...,1,0.0,27.19,1,27.19,5,0,Very Early,0,0
25818,b33ec3b699337181488304f362a6b734,9b3932a6253894a02c1df9d19004239f,0000f46a3911fa3c0805444483337064,2017-03-10 21:05:03,2017-04-05 14:38:47,2017-04-07,3.0,None,None,-2,...,1,0.0,86.22,1,86.22,3,4,Slightly Early,0,0
96079,41272756ecddd9a9ed0180413cc22fb6,914991f0c02ef0843c0e7010c819d642,0000f6ccb0745a6a4b88665a16c9f078,2017-10-12 20:29:41,2017-11-01 21:23:05,2017-11-13,4.0,Bom vendedor,None,-12,...,1,0.0,43.62,1,43.62,10,3,Very Early,0,0
40531,d957021f1127559cd947b62533f484f7,47227568b10f5f58a524a75507e6992c,0004aac84e0df4da2b147fca70cf8255,2017-11-14 19:45:42,2017-11-27 23:08:56,2017-12-05,5.0,None,None,-8,...,1,0.0,196.89,1,196.89,11,1,Very Early,0,0


In [83]:
# Inspection
df[[
    "order_id",
    "customer_unique_id",
    "review_score",
    "has_review",
    "is_dissatisfied",
    "delivery_delay",
    "delivery_deviation_abs",
    "order_value",
    "product_count",
    "avg_price_per_item",
    "prior_orders",
    "is_repeat_customer",
    "purchase_month",
    "purchase_dow",
    "delay_bucket"
]].head()

,order_id,customer_unique_id,review_score,has_review,is_dissatisfied,delivery_delay,delivery_deviation_abs,order_value,product_count,avg_price_per_item,prior_orders,is_repeat_customer,purchase_month,purchase_dow,delay_bucket
51501,e22acc9c116caa3f2b7121bbb380d08e,0000366f3b9a7992bf8c76cfdf3221e2,5.0,1,0.0,-5,5,141.90,1,141.90,0,0,5,3,Very Early
72082,3594e05a005ac4d06a72673270ef9ec9,0000b849f77a49e4a4ce2b2a4ca5be3f,4.0,1,0.0,-5,5,27.19,1,27.19,0,0,5,0,Very Early
25818,b33ec3b699337181488304f362a6b734,0000f46a3911fa3c0805444483337064,3.0,1,0.0,-2,2,86.22,1,86.22,0,0,3,4,Slightly Early
96079,41272756ecddd9a9ed0180413cc22fb6,0000f6ccb0745a6a4b88665a16c9f078,4.0,1,0.0,-12,12,43.62,1,43.62,0,0,10,3,Very Early
40531,d957021f1127559cd947b62533f484f7,0004aac84e0df4da2b147fca70cf8255,5.0,1,0.0,-8,8,196.89,1,196.89,0,0,11,1,Very Early


In [84]:
df[[
    "review_score",
    "has_review",
    "is_dissatisfied",
    "delivery_delay",
    "order_value",
    "product_count",
    "avg_price_per_item",
    "prior_orders"
]].isnull().sum()

review_score          646
has_review              0
is_dissatisfied       646
delivery_delay          0
order_value             0
product_count           0
avg_price_per_item      0
prior_orders            0
dtype: int64

## Final dataset structure

The final modeling dataset retains all delivered orders and distinguishes between:

- **feedback availability** (`has_review`)
- **observed dissatisfaction** (`is_dissatisfied`) for reviewed orders only

This supports both:
1. customer dissatisfaction modeling among reviewed orders
2. behavioral analysis of who leaves feedback versus who remains silent

In [ ]:
#df.to_csv("../data/processed/main_dataset.csv", index=False)